## Цель 

Собрать рабочий baseline pipeline: 
1. чтение изображений из папки;
2. детекция таблички; 
3. OCR номера; 
4. нормализация результата; 
5. расчёт confidence; 
6. формирование нового имени файла; 
7. отчёт по обработке.

## Импорты

In [1]:
from pathlib import Path 
import random 
import re 
import shutil 
import cv2 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 

from ultralytics import YOLO 
import easyocr

## Пути

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATASET_DIR = PROJECT_ROOT / "dataset"
IMAGES_DIR = DATASET_DIR / "images"
MODEL_PATH = PROJECT_ROOT / "runs" / "detect" / "train4" / "weights" / "best.pt"
REPORTS_DIR = PROJECT_ROOT / "metadata"

REPORTS_DIR.mkdir(exist_ok=True)

print("MODEL_PATH exists:", MODEL_PATH.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())

MODEL_PATH exists: True
IMAGES_DIR exists: True


## Загрузка модели и OCR

In [3]:
model = YOLO(str(MODEL_PATH))
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


## Список изображений

In [4]:
image_paths = []

for split in ["train", "val"]:
    split_dir = IMAGES_DIR / split
    image_paths.extend([
        p for p in split_dir.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

print("Total images:", len(image_paths))

Total images: 123


## Базовые функции

In [5]:
def crop_bbox(image_bgr, bbox):
    h, w = image_bgr.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    return image_bgr[y1:y2, x1:x2]


def normalize_plate_orientation(image_bgr):
    h, w = image_bgr.shape[:2]
    if h > w:
        image_bgr = cv2.rotate(image_bgr, cv2.ROTATE_90_CLOCKWISE)
    return image_bgr


def extract_number_zone_wide(plate):
    h, w = plate.shape[:2]
    return plate[
        int(h * 0.35):int(h * 0.85),
        int(w * 0.05):int(w * 0.95)
    ]


def preprocess_blur_only(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)
    return th


def extract_text_from_easyocr(ocr_output):
    texts = []
    confs = []

    for item in ocr_output:
        bbox, text, conf = item
        texts.append(text)
        confs.append(conf)

    merged_text = " ".join(texts)
    mean_conf = float(np.mean(confs)) if confs else 0.0

    return merged_text, mean_conf


def process_ocr_text(raw_text: str) -> str:
    text = raw_text.lower()
    text = re.sub(r"[^0-9abcde]", "", text)

    match = re.fullmatch(r"\d{1,4}[abcde]?", text)
    return match.group(0) if match else ""


def normalize_plate_number(text: str) -> str:
    match = re.fullmatch(r"(\d{1,4})([abcde]?)", text)
    if not match:
        return ""

    digits, suffix = match.groups()
    digits = digits.zfill(4)
    return digits + suffix


def extract_expected_number_from_filename(filename: str) -> str:
    stem = Path(filename).stem.lower()
    match = re.search(r"(\d{1,4}[abcde]?)", stem)
    if not match:
        return ""
    return normalize_plate_number(match.group(1))

## Итоговый pipeline для одного изображения

In [14]:
def run_pipeline(image_path, model, reader):
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        return {
            "filename": image_path.name,
            "detected": False,
            "det_conf": 0.0,
            "ocr_raw": "",
            "ocr_clean": "",
            "ocr_norm": "",
            "ocr_conf": 0.0,
            "final_conf": 0.0,
            "status": "image_read_failed",
            "zone_name": "",
            "prep_name": "",
        }

    det_res = model(str(image_path), verbose=False)[0]

    if len(det_res.boxes) == 0:
        return {
            "filename": image_path.name,
            "detected": False,
            "det_conf": 0.0,
            "ocr_raw": "",
            "ocr_clean": "",
            "ocr_norm": "",
            "ocr_conf": 0.0,
            "final_conf": 0.0,
            "status": "no_detection",
            "zone_name": "",
            "prep_name": "",
        }

    boxes_xyxy = det_res.boxes.xyxy.cpu().numpy()
    confidences = det_res.boxes.conf.cpu().numpy()
    best_idx = np.argmax(confidences)

    bbox = boxes_xyxy[best_idx]
    det_conf = float(confidences[best_idx])

    plate = crop_bbox(image_bgr, bbox)

    if plate.size == 0:
        return {
            "filename": image_path.name,
            "detected": True,
            "det_conf": det_conf,
            "ocr_raw": "",
            "ocr_clean": "",
            "ocr_norm": "",
            "ocr_conf": 0.0,
            "final_conf": 0.0,
            "status": "empty_crop",
            "zone_name": "",
            "prep_name": "",
        }

    plate = normalize_plate_orientation(plate)

    ocr_best = choose_best_zone_and_ocr(reader, plate)

    ocr_raw = ocr_best["ocr_raw"]
    ocr_clean = ocr_best["ocr_clean"]
    ocr_norm = ocr_best["ocr_norm"]
    ocr_conf = ocr_best["ocr_conf"]
    zone_name = ocr_best["zone_name"] or ""
    prep_name = ocr_best["variant_name"] or ""

    final_conf = det_conf * ocr_conf
    status = "ok" if ocr_norm else "ocr_failed"

    return {
        "filename": image_path.name,
        "detected": True,
        "det_conf": det_conf,
        "ocr_raw": ocr_raw,
        "ocr_clean": ocr_clean,
        "ocr_norm": ocr_norm,
        "ocr_conf": ocr_conf,
        "final_conf": final_conf,
        "status": status,
        "zone_name": zone_name,
        "prep_name": prep_name,
    }

## Улучшение OCR: multi-preprocess + allowlist + upscale

## Новые функции предобработки

In [7]:
def make_ocr_variants(gray):
    variants = {}

    variants["gray"] = gray

    variants["binary_130"] = cv2.threshold(
        gray, 130, 255, cv2.THRESH_BINARY
    )[1]

    variants["binary_otsu"] = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    variants["adaptive_gaussian"] = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        10
    )

    variants["inv_otsu"] = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )[1]

    up2 = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    variants["up2_gray"] = up2
    variants["up2_otsu"] = cv2.threshold(
        up2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    up3 = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    variants["up3_otsu"] = cv2.threshold(
        up3, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]

    kernel = np.ones((3, 3), np.uint8)
    variants["morph_close_otsu"] = cv2.morphologyEx(
        variants["binary_otsu"], cv2.MORPH_CLOSE, kernel
    )

    return variants

## Исправление ошибок OCR

In [8]:
def fix_common_ocr_confusions(text: str) -> str:
    text = text.lower()

    mapping = {
        "o": "0",
        "q": "0",
        "i": "1",
        "l": "1",
        "s": "5",
        "z": "2",
    }

    return "".join(mapping.get(ch, ch) for ch in text)

In [9]:
def process_ocr_text_soft(raw_text: str) -> str:
    text = fix_common_ocr_confusions(raw_text)
    text = re.sub(r"[^0-9abcde]", "", text)

    candidates = re.findall(r"\d{1,4}[abcde]?", text)
    if not candidates:
        return ""

    candidates = sorted(candidates, key=len, reverse=True)
    return candidates[0]

In [10]:
def run_easyocr_on_image(reader, image):
    result = reader.readtext(
        image,
        allowlist="0123456789abcdeABCDE"
    )
    raw, conf = extract_text_from_easyocr(result)
    return raw, conf

In [11]:
def choose_best_ocr_candidate(reader, zone_bgr):
    gray = cv2.cvtColor(zone_bgr, cv2.COLOR_BGR2GRAY)
    variants = make_ocr_variants(gray)

    best = {
        "ocr_raw": "",
        "ocr_clean": "",
        "ocr_norm": "",
        "ocr_conf": 0.0,
        "variant_name": None,
        "score": -1.0,
    }

    for variant_name, img in variants.items():
        raw, conf = run_easyocr_on_image(reader, img)
        clean = process_ocr_text_soft(raw)
        norm = normalize_plate_number(clean)

        score = float(conf)

        if clean:
            score += 0.2

        if norm:
            score += 1.0

        if score > best["score"]:
            best = {
                "ocr_raw": raw,
                "ocr_clean": clean,
                "ocr_norm": norm,
                "ocr_conf": float(conf),
                "variant_name": variant_name,
                "score": score,
            }

    return best

In [12]:
def extract_number_zone_variants(plate):
    h, w = plate.shape[:2]

    variants = {}

    variants["wide"] = plate[
        int(h * 0.35):int(h * 0.85),
        int(w * 0.05):int(w * 0.95)
    ]

    variants["center"] = plate[
        int(h * 0.30):int(h * 0.80),
        int(w * 0.12):int(w * 0.88)
    ]

    variants["tight"] = plate[
        int(h * 0.40):int(h * 0.78),
        int(w * 0.15):int(w * 0.85)
    ]

    return variants

In [13]:
def choose_best_zone_and_ocr(reader, plate_bgr):
    zone_variants = extract_number_zone_variants(plate_bgr)

    best = {
        "ocr_raw": "",
        "ocr_clean": "",
        "ocr_norm": "",
        "ocr_conf": 0.0,
        "zone_name": None,
        "variant_name": None,
        "score": -1.0,
    }

    for zone_name, zone in zone_variants.items():
        if zone.size == 0:
            continue

        candidate = choose_best_ocr_candidate(reader, zone)
        score = candidate["score"]

        if score > best["score"]:
            best = {
                "ocr_raw": candidate["ocr_raw"],
                "ocr_clean": candidate["ocr_clean"],
                "ocr_norm": candidate["ocr_norm"],
                "ocr_conf": candidate["ocr_conf"],
                "zone_name": zone_name,
                "variant_name": candidate["variant_name"],
                "score": score,
            }

    return best

In [15]:
random.seed(42)
sample_paths = random.sample(image_paths, min(20, len(image_paths)))

rows = []

for path in sample_paths:
    res = run_pipeline(path, model, reader)
    expected = extract_expected_number_from_filename(path.name)

    res["expected"] = expected
    res["match"] = res["ocr_norm"] == expected if res["ocr_norm"] and expected else False

    rows.append(res)

df_sample = pd.DataFrame(rows)
df_sample

/home/alexander/venvs/plate-recognition/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/alexander/venvs/plate-recognition/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,zone_name,prep_name,expected,match
0,MDZ090 _9.jpg,True,0.921437,0,0,0000,0.996095,0.917840,ok,center,up2_gray,0090,False
1,TUL864_1.jpg,True,0.879056,8 6 4,864,0864,0.999999,0.879055,ok,tight,up2_gray,0864,True
2,TUL803_3.jpg,True,0.948048,3,3,0003,0.999991,0.948040,ok,center,adaptive_gaussian,0803,False
3,MDZ097_3.jpg,True,0.889849,9 7,97,0097,0.999994,0.889843,ok,tight,up3_otsu,0097,True
4,TUL082_6.jpg,True,0.942360,8 2,82,0082,1.000000,0.942360,ok,center,gray,0082,True
5,TUL051_2.jpg,True,0.905069,5 1,51,0051,1.000000,0.905069,ok,tight,inv_otsu,0051,True
6,TUL023_6.jpg,True,0.984962,23,23,0023,0.999999,0.984961,ok,center,inv_otsu,0023,True
7,TUL866.jpg,True,0.964316,8 6 6,866,0866,0.999999,0.964314,ok,tight,up2_gray,0866,True
8,TUL859.jpg,True,0.860519,859,859,0859,1.000000,0.860519,ok,wide,up3_otsu,0859,True
9,MDZ094 _4.jpg,True,0.928574,9 4,94,0094,1.000000,0.928574,ok,center,gray,0094,True


In [16]:
df_sample[[
    "filename", "expected", "ocr_norm", "match",
    "det_conf", "ocr_conf", "final_conf",
    "zone_name", "prep_name", "status"
]]

,filename,expected,ocr_norm,match,det_conf,ocr_conf,final_conf,zone_name,prep_name,status
0,MDZ090 _9.jpg,0090,0000,False,0.921437,0.996095,0.917840,center,up2_gray,ok
1,TUL864_1.jpg,0864,0864,True,0.879056,0.999999,0.879055,tight,up2_gray,ok
2,TUL803_3.jpg,0803,0003,False,0.948048,0.999991,0.948040,center,adaptive_gaussian,ok
3,MDZ097_3.jpg,0097,0097,True,0.889849,0.999994,0.889843,tight,up3_otsu,ok
4,TUL082_6.jpg,0082,0082,True,0.942360,1.000000,0.942360,center,gray,ok
5,TUL051_2.jpg,0051,0051,True,0.905069,1.000000,0.905069,tight,inv_otsu,ok
6,TUL023_6.jpg,0023,0023,True,0.984962,0.999999,0.984961,center,inv_otsu,ok
7,TUL866.jpg,0866,0866,True,0.964316,0.999999,0.964314,tight,up2_gray,ok
8,TUL859.jpg,0859,0859,True,0.860519,1.000000,0.860519,wide,up3_otsu,ok
9,MDZ094 _4.jpg,0094,0094,True,0.928574,1.000000,0.928574,center,gray,ok


In [18]:
df_sample["match"].mean()

np.float64(0.55)

## Выводы


In [21]:
print("=== Общая статистика ===")
print("Accuracy (match):", float(df_sample["match"].mean()))
print("Total samples:", len(df_sample))


print("\n=== Ошибки (match == False) ===")
display(
    df_sample[df_sample["match"] == False][[
        "filename",
        "expected",
        "ocr_norm",
        "ocr_raw",
        "ocr_clean",
        "det_conf",
        "ocr_conf",
        "final_conf",
        "zone_name",
        "prep_name",
        "status"
    ]]
)


print("\n=== High confidence ошибки ===")
display(
    df_sample[
        (df_sample["match"] == False)
    ][[
        "filename",
        "expected",
        "ocr_norm",
        "ocr_raw",
        "ocr_clean",
        "det_conf",
        "ocr_conf",
        "final_conf",
        "zone_name",
        "prep_name"
    ]]
)


print("\n=== Топ сомнительных (low final_conf) ===")
display(
    df_sample.sort_values(by="final_conf", ascending=True).head(10)[[
        "filename",
        "expected",
        "ocr_norm",
        "match",
        "final_conf",
        "zone_name",
        "prep_name"
    ]]
)


print("\n=== Распределение confidence ===")
display(
    df_sample[[
        "det_conf",
        "ocr_conf",
        "final_conf"
    ]].describe()
)

=== Общая статистика ===
Accuracy (match): 0.55
Total samples: 20

=== Ошибки (match == False) ===


,filename,expected,ocr_norm,ocr_raw,ocr_clean,det_conf,ocr_conf,final_conf,zone_name,prep_name,status
0,MDZ090 _9.jpg,0090,0000,0,0,0.921437,0.996095,0.917840,center,up2_gray,ok
2,TUL803_3.jpg,0803,0003,3,3,0.948048,0.999991,0.948040,center,adaptive_gaussian,ok
11,QBA1156_5.jpg,1156,0099,9 9,99,0.914066,0.968833,0.885577,center,adaptive_gaussian,ok
12,TUL855_5.jpg,0855,0008,8,8,0.585546,1.000000,0.585546,tight,binary_130,ok
13,MDZ087 _3.jpg,0087,0187,187,187,0.918648,0.999153,0.917870,wide,morph_close_otsu,ok
15,TUL831_4.jpg,0831,0083,83,83,0.988873,0.999999,0.988872,center,binary_otsu,ok
16,TUL021_1.jpg,0021,0002,2,2,0.935153,1.000000,0.935153,center,adaptive_gaussian,ok
18,QBA1096a_2.jpg,1096a,0009,09,09,0.835561,0.999646,0.835265,center,up2_gray,ok
19,MDZ087 _8.jpg,0087,8107,8 10 7,8107,0.927897,0.972084,0.901994,tight,up2_otsu,ok



=== High confidence ошибки ===


,filename,expected,ocr_norm,ocr_raw,ocr_clean,det_conf,ocr_conf,final_conf,zone_name,prep_name
0,MDZ090 _9.jpg,0090,0000,0,0,0.921437,0.996095,0.917840,center,up2_gray
2,TUL803_3.jpg,0803,0003,3,3,0.948048,0.999991,0.948040,center,adaptive_gaussian
11,QBA1156_5.jpg,1156,0099,9 9,99,0.914066,0.968833,0.885577,center,adaptive_gaussian
12,TUL855_5.jpg,0855,0008,8,8,0.585546,1.000000,0.585546,tight,binary_130
13,MDZ087 _3.jpg,0087,0187,187,187,0.918648,0.999153,0.917870,wide,morph_close_otsu
15,TUL831_4.jpg,0831,0083,83,83,0.988873,0.999999,0.988872,center,binary_otsu
16,TUL021_1.jpg,0021,0002,2,2,0.935153,1.000000,0.935153,center,adaptive_gaussian
18,QBA1096a_2.jpg,1096a,0009,09,09,0.835561,0.999646,0.835265,center,up2_gray
19,MDZ087 _8.jpg,0087,8107,8 10 7,8107,0.927897,0.972084,0.901994,tight,up2_otsu



=== Топ сомнительных (low final_conf) ===


,filename,expected,ocr_norm,match,final_conf,zone_name,prep_name
12,TUL855_5.jpg,0855,0008,False,0.585546,tight,binary_130
18,QBA1096a_2.jpg,1096a,0009,False,0.835265,center,up2_gray
8,TUL859.jpg,0859,0859,True,0.860519,wide,up3_otsu
1,TUL864_1.jpg,0864,0864,True,0.879055,tight,up2_gray
11,QBA1156_5.jpg,1156,0099,False,0.885577,center,adaptive_gaussian
3,MDZ097_3.jpg,0097,0097,True,0.889843,tight,up3_otsu
19,MDZ087 _8.jpg,0087,8107,False,0.901994,tight,up2_otsu
5,TUL051_2.jpg,0051,0051,True,0.905069,tight,inv_otsu
0,MDZ090 _9.jpg,0090,0000,False,0.917840,center,up2_gray
13,MDZ087 _3.jpg,0087,0187,False,0.917870,wide,morph_close_otsu



=== Распределение confidence ===


,det_conf,ocr_conf,final_conf
count,20.000000,20.000000,20.000000
mean,0.911420,0.996789,0.908466
std,0.086941,0.009063,0.087025
min,0.585546,0.968833,0.585546
25%,0.901264,0.999905,0.888777
50%,0.928236,0.999999,0.923222
75%,0.952115,1.000000,0.952109
max,0.988873,1.000000,0.988872
